# OpenAI Agents SDK: Handoffs, Guardrails y Multi-Agente

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/agentes-avanzados/11-openai-agents-sdk.ipynb)

Construcción de sistemas multi-agente con OpenAI Agents SDK: handoffs entre agentes especializados, guardrails y streaming.

**Contenidos:**
1. Instalación y primitivas del SDK
2. Agent básico con herramientas
3. Handoff entre dos agentes
4. Guardrail de entrada (InputGuardrail)
5. Sistema triage con 3 agentes especializados
6. Streaming con Runner.run_streamed()
7. Comparativa: OpenAI Agents SDK vs Claude Agent SDK vs LangGraph

In [ ]:
# Instalación
# !pip install openai-agents

import os
import asyncio
from typing import Optional
from pydantic import BaseModel

# OpenAI Agents SDK
from agents import Agent, Runner, handoff, tool, RunResult
from agents.guardrails import InputGuardrail, OutputGuardrail, GuardrailFunctionOutput
from agents.exceptions import GuardrailTripwireTriggered

# Configurar API key
# os.environ["OPENAI_API_KEY"] = "sk-..."

MODEL = "gpt-4o-mini"  # Modelo económico para demos

print("OpenAI Agents SDK importado correctamente.")
print(f"Modelo por defecto: {MODEL}")

## 1. Agent Básico con Herramientas

Un `Agent` combina instrucciones (system prompt) con herramientas Python decoradas con `@tool`.

In [ ]:
import datetime

# --- Definición de herramientas ---

@tool
def obtener_hora_actual(zona_horaria: str = "Europe/Madrid") -> str:
    """Devuelve la hora actual en la zona horaria especificada."""
    ahora = datetime.datetime.now()
    return f"Hora actual en {zona_horaria}: {ahora.strftime('%H:%M:%S del %d/%m/%Y')}"


@tool
def buscar_producto(nombre: str) -> list[dict]:
    """
    Busca productos en el catálogo por nombre.
    
    Args:
        nombre: Nombre o parte del nombre del producto
    """
    catalogo = [
        {"id": "P001", "nombre": "Laptop Pro 15", "precio": 1299.99, "stock": 5},
        {"id": "P002", "nombre": "Laptop Air 13", "precio": 899.99, "stock": 12},
        {"id": "P003", "nombre": "Mouse Inalámbrico", "precio": 29.99, "stock": 50},
        {"id": "P004", "nombre": "Teclado Mecánico", "precio": 89.99, "stock": 8},
        {"id": "P005", "nombre": 'Monitor 4K 27"', "precio": 399.99, "stock": 3},
    ]
    nombre_lower = nombre.lower()
    return [p for p in catalogo if nombre_lower in p["nombre"].lower()]


@tool
def calcular_precio_con_iva(precio_base: float, tipo_iva: float = 21.0) -> dict:
    """
    Calcula el precio final con IVA.
    
    Args:
        precio_base: Precio sin IVA en euros
        tipo_iva: Porcentaje de IVA (default 21%)
    """
    iva = precio_base * (tipo_iva / 100)
    return {
        "precio_base": round(precio_base, 2),
        "iva_pct": tipo_iva,
        "iva_euros": round(iva, 2),
        "total": round(precio_base + iva, 2)
    }


# --- Agente básico ---
agente_ventas = Agent(
    name="AgenteVentas",
    instructions="""Eres un asistente de ventas para una tienda de tecnología.
Ayudas a los clientes a encontrar productos, comparar opciones y conocer precios con IVA.
Usa siempre las herramientas disponibles antes de inventar información.
Sé amable, conciso y en el mismo idioma que el cliente.""",
    tools=[buscar_producto, calcular_precio_con_iva, obtener_hora_actual],
    model=MODEL
)


# Ejecutar el agente
async def demo_agente_basico():
    resultado = await Runner.run(
        agente_ventas,
        "Hola, busco un laptop. ¿Qué tienes disponible y cuánto costaría el más barato con IVA?"
    )
    print(f"Agente: {resultado.last_agent.name}")
    print(f"Respuesta:\n{resultado.final_output}")

await demo_agente_basico()

## 2. Handoff Entre Dos Agentes

El handoff transfiere el control a un agente especializado. El contexto de conversación se preserva.

In [ ]:
@tool
def consultar_estado_pedido(numero_pedido: str) -> dict:
    """Consulta el estado de un pedido por su número."""
    pedidos = {
        "12345": {"estado": "En tránsito", "fecha_entrega": "23/04/2026", "transportista": "SEUR"},
        "12346": {"estado": "Entregado", "fecha_entrega": "18/04/2026", "transportista": "DHL"},
    }
    return pedidos.get(numero_pedido, {"error": f"Pedido {numero_pedido} no encontrado"})


# Agente especializado en pedidos
agente_pedidos = Agent(
    name="EspecialistaPedidos",
    instructions="""Especialista en gestión de pedidos. Consultas estado, devoluciones, entregas.
SIEMPRE solicita el número de pedido antes de buscar.
Si el cliente quiere comprar algo nuevo, transfiere a Ventas.""",
    tools=[consultar_estado_pedido],
    model=MODEL
)

# Agente de triage con handoff a los dos especialistas
agente_triage = Agent(
    name="Triage",
    instructions="""Primer punto de contacto. Solo identificas y derivas:
- Compras y catálogo → AgenteVentas
- Pedidos, entregas, devoluciones → EspecialistaPedidos
No resuelvas consultas tú mismo. Saluda e informa al cliente a quién lo derivas.""",
    handoffs=[
        handoff(agente_ventas),
        handoff(agente_pedidos),
    ],
    model=MODEL
)


async def demo_handoff():
    casos = [
        ("¿Cuánto cuesta el Monitor 4K?", "ventas"),
        ("¿Dónde está mi pedido 12345?", "pedidos"),
    ]
    
    for consulta, tipo_esperado in casos:
        print(f"\nCliente: {consulta}")
        resultado = await Runner.run(agente_triage, consulta)
        print(f"Agente que respondió: {resultado.last_agent.name}")
        print(f"Respuesta: {resultado.final_output[:200]}..." if len(resultado.final_output) > 200 else f"Respuesta: {resultado.final_output}")

await demo_handoff()

## 3. Guardrail de Entrada

Un `InputGuardrail` evalúa la entrada antes de que el agente principal la procese. Si el guardrail activa el `tripwire`, la petición es bloqueada.

In [ ]:
class ClasificacionConsulta(BaseModel):
    """Resultado de clasificar si la consulta es apropiada."""
    es_apropiada: bool
    categoria: str  # "apropiada" | "off_topic" | "inapropiada"
    razon: str


agente_clasificador = Agent(
    name="ClasificadorConsultas",
    instructions="""Clasifica si la consulta es apropiada para una tienda de tecnología.
- apropiada: relacionada con productos, pedidos, soporte técnico de la tienda
- off_topic: preguntas no relacionadas (política, medicina, cocina, etc.)
- inapropiada: lenguaje ofensivo, spam, amenazas

Responde en JSON con: es_apropiada (bool), categoria (str), razon (str)""",
    output_type=ClasificacionConsulta,
    model=MODEL
)


async def guardrail_entrada_fn(ctx, agent: Agent, input_data: str) -> GuardrailFunctionOutput:
    """Evalúa si la entrada es apropiada antes de procesarla."""
    resultado = await Runner.run(agente_clasificador, input_data)
    clasificacion: ClasificacionConsulta = resultado.final_output
    
    if not clasificacion.es_apropiada:
        print(f"  [Guardrail] Bloqueado ({clasificacion.categoria}): {clasificacion.razon}")
    
    return GuardrailFunctionOutput(
        output_info=clasificacion,
        tripwire_triggered=not clasificacion.es_apropiada
    )


# Agente con guardrail de entrada
agente_protegido = Agent(
    name="TriageProtegido",
    instructions=agente_triage.instructions,
    handoffs=agente_triage.handoffs,
    input_guardrails=[InputGuardrail(guardrail_function=guardrail_entrada_fn)],
    model=MODEL
)


async def demo_guardrail():
    consultas = [
        "¿Tenéis laptops gaming?",                    # Apropiada
        "¿Cuál es la capital de Australia?",           # Off-topic
        "¿Me puedes dar una receta de paella?",         # Off-topic
    ]
    
    for consulta in consultas:
        print(f"\nConsulta: '{consulta}'")
        try:
            resultado = await Runner.run(agente_protegido, consulta)
            print(f"Respuesta: {resultado.final_output[:150]}")
        except GuardrailTripwireTriggered:
            print("  BLOQUEADO: consulta fuera de ámbito o inapropiada.")

await demo_guardrail()

## 4. Sistema Triage con 3 Agentes Especializados

Sistema completo: Triage → Ventas / Pedidos / Soporte Técnico, con guardrail integrado.

In [ ]:
@tool
def diagnosticar_problema(modelo: str, sistema_operativo: str, descripcion: str) -> dict:
    """Proporciona diagnóstico inicial para problemas técnicos."""
    return {
        "diagnostico": "Posible conflicto de drivers o actualización reciente",
        "pasos": [
            "1. Reiniciar en modo seguro",
            "2. Ejecutar sfc /scannow en CMD como administrador",
            "3. Verificar actualizaciones pendientes en Device Manager"
        ],
        "ticket_id": "TKT-2026-0001",
        "escalacion": False
    }


# Tres agentes especializados
esp_ventas = Agent(
    name="EspecialistaVentas",
    instructions="""Especialista en ventas. Ayudas a encontrar el producto adecuado.
Siempre pregunta: ¿para qué usará el producto? y ¿cuál es su presupuesto?
Ofrece 2-3 opciones con pros y contras.""",
    tools=[buscar_producto, calcular_precio_con_iva],
    model=MODEL
)

esp_pedidos = Agent(
    name="EspecialistaPedidos",
    instructions="""Especialista en pedidos y post-venta.
Gestiona: estado de pedidos, devoluciones, reclamaciones.
SIEMPRE solicita el número de pedido antes de consultar.""",
    tools=[consultar_estado_pedido],
    model=MODEL
)

esp_tecnico = Agent(
    name="EspecialistaTecnico",
    instructions="""Especialista en soporte técnico.
Protocolo: pide modelo, sistema operativo y descripción del problema antes de diagnosticar.""",
    tools=[diagnosticar_problema],
    model=MODEL
)

# Triage principal con guardrail y 3 handoffs
triage_completo = Agent(
    name="Recepcionista",
    instructions="""Recepcionista virtual de TechStore.
Saluda y redirige cada consulta al especialista correcto:
- Compras y catálogo → EspecialistaVentas
- Pedidos, entregas, devoluciones → EspecialistaPedidos
- Problemas técnicos → EspecialistaTecnico
Al derivar, informa: 'Te paso con [nombre], que te ayudará con [tema]'.""",
    handoffs=[
        handoff(esp_ventas),
        handoff(esp_pedidos),
        handoff(esp_tecnico),
    ],
    input_guardrails=[InputGuardrail(guardrail_function=guardrail_entrada_fn)],
    model=MODEL
)


async def demo_sistema_completo():
    casos = [
        "¿Tenéis laptops por menos de 1000€ para diseño gráfico?",
        "¿Dónde está mi pedido 12346?",
        "Mi laptop no reconoce el WiFi desde la última actualización",
    ]
    
    for caso in casos:
        print(f"\n{'='*60}")
        print(f"Cliente: {caso}")
        try:
            resultado = await Runner.run(triage_completo, caso)
            print(f"Agente: {resultado.last_agent.name}")
            respuesta = resultado.final_output
            print(f"Respuesta: {respuesta[:300]}" + ("..." if len(respuesta) > 300 else ""))
        except GuardrailTripwireTriggered:
            print("BLOQUEADO por guardrail")

await demo_sistema_completo()

## 5. Streaming con Runner.run_streamed()

Para interfaces de usuario donde el texto debe aparecer progresivamente.

In [ ]:
async def demo_streaming():
    """Muestra la respuesta token a token conforme se genera."""
    
    consulta = "Explícame las diferencias entre todos vuestros laptops disponibles y cuál me recomiendas para trabajo de oficina"
    print(f"Consulta: {consulta}\n")
    print("Respuesta (streaming):")
    print("-" * 50)
    
    result = Runner.run_streamed(esp_ventas, consulta)
    
    async for event in result.stream_events():
        if event.type == "raw_response_event":
            # Tokens de texto conforme llegan
            if hasattr(event.data, "delta") and hasattr(event.data.delta, "text"):
                print(event.data.delta.text, end="", flush=True)
        
        elif event.type == "agent_updated_stream_event":
            # Notificación de handoff
            print(f"\n[→ Handoff a {event.new_agent.name}]\n")
    
    print("\n" + "-" * 50)
    print(f"Agente final: {result.current_agent.name}")

await demo_streaming()

## 6. Comparativa con Otras Alternativas

In [ ]:
comparativa = """
╔═══════════════════╦══════════════════╦══════════════════╦══════════════════╗
║ Característica    ║ OpenAI Agents SDK║  Claude Agent SDK║    LangGraph     ║
╠═══════════════════╬══════════════════╬══════════════════╬══════════════════╣
║ Proveedor         ║ OpenAI           ║ Anthropic        ║ LangChain        ║
║ Modelos           ║ Solo GPT         ║ Solo Claude      ║ Cualquiera       ║
║ Curva aprendizaje ║ Baja             ║ Baja             ║ Media-Alta       ║
║ Handoffs          ║ Nativo simple    ║ Nativo           ║ Via edges        ║
║ Guardrails        ║ Nativo           ║ Via tools        ║ Manual           ║
║ Ciclos complejos  ║ Limitado         ║ Limitado         ║ Excelente        ║
║ Visualización     ║ No               ║ No               ║ Sí               ║
║ Persistencia      ║ Manual           ║ Manual           ║ LangGraph Platf. ║
║ Streaming         ║ Nativo           ║ Nativo           ║ Sí               ║
║ Multi-proveedor   ║ No               ║ No               ║ Sí               ║
╚═══════════════════╩══════════════════╩══════════════════╩══════════════════╝

Cuándo elegir cada uno:
• OpenAI Agents SDK  → Ya usas GPT, quieres handoffs simples con mínimo código
• Claude Agent SDK   → Necesitas Extended Thinking o capacidades avanzadas de Claude
• LangGraph          → Flujos complejos, múltiples proveedores, trazabilidad completa
"""
print(comparativa)